# SQL commands - 

## Remove Existing Table
- DROP TABLE IF EXISTS sales;

## Create Table
- CREATE TABLE sales (
    id         INT PRIMARY KEY,
    salesperson VARCHAR(100) NOT NULL,
    region      VARCHAR(50)  NOT NULL,
    sale_date   DATE         NOT NULL,
    amount      DECIMAL(10,2) NOT NULL
);

## Sample Data
- INSERT INTO sales (id, salesperson, region, sale_date, amount) VALUES
(1, 'Alice', 'East', '2025-08-01', 200),
(2, 'Bob',   'East', '2025-08-02', 150),
(3, 'Alice', 'West', '2025-08-03', 300),
(4, 'Bob',   'East', '2025-08-04', 250),
(5, 'Alice', 'East', '2025-08-05', 100),
(6, 'Bob',   'East', '2025-08-06', 250),   -- tie with Bob's 250
(7, 'Carol', 'East', '2025-08-02', 150),   -- tie with Bob's 150
(8, 'Dan',   'West', '2025-08-04', 300);   -- tie with Alice's 300 (West)

## Check
- SELECT * FROM sales ORDER BY id;

## VIEW1
### Region-level summary (non-updatable: uses aggregates)
- CREATE OR REPLACE VIEW region_sales_summary AS
SELECT
    region,
    SUM(amount) AS total_sales,
    COUNT(*)    AS num_sales,
    AVG(amount) AS avg_amount
FROM sales
GROUP BY region;
#### Use the View
- SELECT * FROM region_sales_summary ORDER BY region;

### VIEW2
#### Salesperson-level KPIs (non-updatable: aggregates)
- CREATE OR REPLACE VIEW salesperson_totals AS
SELECT
    salesperson,
    SUM(amount) AS total_sales,
    COUNT(*)    AS num_sales,
    MIN(sale_date) AS first_sale_date,
    MAX(sale_date) AS last_sale_date
FROM sales
GROUP BY salesperson;
#### See the View
- SELECT * FROM salesperson_totals ORDER BY total_sales DESC;

###  read/update through this view (example read):
- SHOW FULL TABLES WHERE Table_type = 'VIEW';
- CREATE OR REPLACE VIEW east_region_sales_summary AS SELECT id, salesperson, region, sale_date ,SUM(amount) AS total_sales, COUNT(*) AS num_sales FROM sales where   region ='east'  GROUP BY id, salesperson, region, sale_date;
- CREATE OR REPLACE VIEW east_sales AS SELECT * FROM sales where region ='east';
- SELECT * FROM east_sales ORDER BY sale_date;
##### update via updatable view (optional; comment out if not desired):
- UPDATE east_sales SET amount = amount + 10 WHERE id = 2;
- SELECT * FROM east_sales WHERE id = 2;

### View defined over a subquery (inline view inside the view)
- CREATE OR REPLACE VIEW top_sellers AS
SELECT *
FROM (
    SELECT salesperson, SUM(amount) AS total_sales
    FROM sales
    GROUP BY salesperson
) t
WHERE t.total_sales > 300;
#### See the View
- SELECT * FROM top_sellers ORDER BY total_sales DESC;

## When to use which?
- Summarizing only? Use GROUP BY.
- Details + Totals? Use PARTITION BY.(window function)
- Filtering by a summary value? Use a Subquery or CTE (Common Table Expression).
- Use a Subquery for very short, single-line filters (e.g., WHERE price > (SELECT AVG(price) FROM products)).
- Use a CTE if you need to join the same temporary result twice, or if your query has more than two "layers" of logic.
- Use a Temp Table if the CTE or subquery is very large and you need to add an index to it for better performance.
- Use EXISTS for "Yes/No" logic across tables.


# SUBQUERIES
## 2A - Subquery in WHERE/HAVING:
### `Salespeople whose total sales exceed the overall average sale amount`
`To calculate an overall average when people appear multiple times, you must first collapse the data to ensure each person is counted only once, either by averaging their individual records or using distinct counts. The best approach is to compute the average of each person's values first, then take the average of those results. sales table is this kind of table.` 
  
~~SELECT salesperson
FROM sales
GROUP BY salesperson
HAVING SUM(amount) > (
    SELECT AVG(amount)
    FROM sales
);~~  This code will work if there is no duplicate persons.

SELECT salesperson FROM sales GROUP BY salesperson HAVING SUM(amount) > ( SELECT AVG(total_sales) FROM ( SELECT SUM(amount) AS total_sales    FROM sales GROUP BY salesperson ) AS salesperson_totals);

## 2B  Subquery in FROM (inline view):
### `Build per-person totals, then filter in the outer query( salespersons having sales more than 300)`
- SELECT *
FROM (
    SELECT salesperson, SUM(amount) AS total_sales
    FROM sales
    GROUP BY salesperson
) AS sales_totals
WHERE total_sales > 300
ORDER BY total_sales DESC; `with subquery.`

- SELECT salesperson, SUM(amount) AS total_sales FROM sales GROUP BY salesperson having total_sales > 300 ORDER BY total_sales DESC; `(same result without subquery)`


## 2C Subquery in SELECT (scalar correlated subquery)
### `Show each sale alongside the average amount for its region`
- SELECT
    s1.id,
    s1.salesperson,
    s1.region,
    s1.amount,
    (
        SELECT AVG(s2.amount)
        FROM sales AS s2
        WHERE s2.region = s1.region
    ) AS avg_in_region
FROM sales AS s1
ORDER BY s1.region, s1.amount DESC;

#### `With partition`(Same result)

SELECT 
    salesperson,
    region,
    amount,
    AVG(amount) OVER(PARTITION BY region) AS regional_average
FROM 
    sales
ORDER BY 
    region, 
    amount DESC;

## 2D EXISTS (semi-join style):
### `Salespeople who have at least one sale > the average for their region.` 
#### Using Exists:
SELECT DISTINCT s1.salesperson
FROM sales AS s1
WHERE EXISTS (
    SELECT 1
    FROM sales AS s2
    WHERE s2.region = s1.region
    GROUP BY s2.region
    HAVING MAX(s1.amount) > AVG(s2.amount)
)
ORDER BY s1.salesperson;

#### Window function (same result faster in this case)

SELECT DISTINCT salesperson
FROM (
    SELECT 
        salesperson,
        amount > AVG(amount) OVER(PARTITION BY region) AS is_above_avg
    FROM sales
) sub
WHERE is_above_avg = 1
ORDER BY salesperson;

The query works in three distinct stages:
- Subquery (Window Calculation): It first scans the sales table. For every row, it calculates the regional average using AVG(amount) OVER(PARTITION BY region). It then creates a flag (is_above_avg) that is 1 (true) if that specific sale's amount is higher than its regional average. If a sale is 1,200 and the regional average is 1,000, is_above_avg becomes 1.The Filter: The outer query then simply looks for WHERE is_above_avg = 1 to find the "winners."
- Filtering: The outer query filters for only those rows where is_above_avg = 1.
- Deduplication & Sorting: Because a salesperson might have multiple sales above the average, SELECT DISTINCT ensures each name appears only once in the final list. ORDER BY then sorts them alphabetically.

### `Salesperson who have made at least one sale over $299.` 
Exists works best in this case, beacuse when condition is specific, and when satisfied it will stop. No need to scan entire database.

SELECT DISTINCT s1.salesperson
FROM sales AS s1
WHERE EXISTS (
    SELECT 1
    FROM sales AS s2
    WHERE s2.region = s1.region
    GROUP BY s2.region
    HAVING MAX(s1.amount) >299
)
ORDER BY s1.salesperson;

## 2E Correlated subquery in WHERE:
### `Latest sale (by date) for each salesperson`

SELECT *
FROM sales AS s1
WHERE s1.sale_date = (
    SELECT MAX(s2.sale_date)
    FROM sales AS s2
    WHERE s2.salesperson = s1.salesperson
)
ORDER BY salesperson, sale_date;